In [ ]:
# Imports
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# Reusable framework code (installed via `pip install -e .` from the repo root)
from soyini import config
from soyini.classification import (
    normalize_years,
    zscore_by_group,
    assign_cluster_names_from_centers,
)
from soyini.plotting import (
    plot_classification_scatter,
    plot_classification_heatmap,
)

In [ ]:
# Configuration (paths resolved by soyini.config)
casr_input_dir = config.daily_all_csv()   # input: SWEI/Alberta_casr_daily_all_new.csv
drought_subset_dir = config.output_data("SWEI", "Drought_Years_by_Elevation.csv", ensure=False)
shapefile = config.elevation_shapefile()
output_dir = config.output_data("Classification")
output_plots = config.output_plots("Classification")

In [ ]:
# load the CASR data
casr_data = pd.read_csv(casr_input_dir)
display(casr_data.head())

In [4]:
# find peak SWE per each Grid point and Seasonala_Year
peak_idx = (
    casr_data
    .loc[casr_data['SWE'].notna()]
    .groupby(['Grid_ID', 'Seasonal_Year'])['SWE']
    .idxmax()
)

peak_SWE_data = (
    casr_data
    .loc[peak_idx, ['Grid_ID', 'elev_class','lon','lat','Seasonal_Year', 'time', 'SWE']]
    .rename(columns={'time': 'Peak_SWE_Date', 'SWE': 'Peak_SWE'})
    .reset_index(drop=True)
)

display(peak_SWE_data.head())

,Grid_ID,elev_class,lon,lat,Seasonal_Year,Peak_SWE_Date,Peak_SWE
0,1823,1000_1500m,-112.4088,49.0028,1980,1981-02-08,24.4375
1,1823,1000_1500m,-112.4088,49.0028,1981,1982-03-25,46.2500
2,1823,1000_1500m,-112.4088,49.0028,1982,1982-11-21,17.8750
3,1823,1000_1500m,-112.4088,49.0028,1983,1983-12-31,33.5000
4,1823,1000_1500m,-112.4088,49.0028,1984,1985-03-03,40.3750


In [5]:
# calculate meanpeak SWE per Elevation category and Seasonal Year
mean_peak_SWE = (
    peak_SWE_data
    .groupby(['elev_class', 'Seasonal_Year'], as_index=False)
    .agg(Mean_Peak_SWE=('Peak_SWE', 'mean'))
)
display(mean_peak_SWE.head())

,elev_class,Seasonal_Year,Mean_Peak_SWE
0,0_500m,1980,103.993157
1,0_500m,1981,113.597388
2,0_500m,1982,136.846890
3,0_500m,1983,79.861341
4,0_500m,1984,132.457751


In [6]:
# compute climatological peak SWE per Elevation category 1991-2020
clim_peak_SWE = (
    mean_peak_SWE
    .loc[mean_peak_SWE['Seasonal_Year'].between(1991, 2020)]
    .groupby('elev_class', as_index=False)
    .agg(Climatological_Peak_SWE=('Mean_Peak_SWE', 'mean'))
)   
display(clim_peak_SWE.head())

,elev_class,Climatological_Peak_SWE
0,0_500m,107.367150
1,1000_1500m,62.826138
2,1500_2000m,116.367265
3,2000_2500m,168.419924
4,500_1000m,75.548293


In [7]:
# 5% of climatological peak SWE per Elevation category
clim_peak_SWE['onset_SWE'] = clim_peak_SWE['Climatological_Peak_SWE'] * 0.05
display(clim_peak_SWE.head())

,elev_class,Climatological_Peak_SWE,onset_SWE
0,0_500m,107.367150,5.368357
1,1000_1500m,62.826138,3.141307
2,1500_2000m,116.367265,5.818363
3,2000_2500m,168.419924,8.420996
4,500_1000m,75.548293,3.777415


In [8]:
# take cum sum of Precipitation and mean SWE over each Seasonal_Year, per grid
SD_clasification_data = casr_data.sort_values(
    ['Grid_ID', 'Seasonal_Year', 'time']
).copy()

SD_clasification_data['Cumulative_Precipitation'] = (
    SD_clasification_data
    .groupby(['Grid_ID', 'Seasonal_Year'])['Precipitation']
    .cumsum()
)

# get max cumulative Precipitation and mean SWE for each grid and Seasonal_Year
seasonal_maxcumP_meanSWE = (
    SD_clasification_data
    .groupby(['Grid_ID', 'Seasonal_Year', 'lon', 'lat', 'elev_class'], as_index=False)
    .agg(
        Max_Cumulative_Precipitation=('Cumulative_Precipitation', 'max'),
        Mean_SWE=('SWE', 'mean')
    )
)

display(seasonal_maxcumP_meanSWE.head())

,Grid_ID,Seasonal_Year,lon,lat,elev_class,Max_Cumulative_Precipitation,Mean_SWE
0,1823,1980,-112.4088,49.0028,1000_1500m,233.992671,3.701920
1,1823,1981,-112.4088,49.0028,1000_1500m,221.355045,12.164492
2,1823,1982,-112.4088,49.0028,1000_1500m,130.209265,4.008709
3,1823,1983,-112.4088,49.0028,1000_1500m,146.054043,5.082673
4,1823,1984,-112.4088,49.0028,1000_1500m,218.806514,11.118445


In [9]:
mean_cumP_by_elev = seasonal_maxcumP_meanSWE.groupby('elev_class')['Max_Cumulative_Precipitation'].mean().reset_index()
mean_cumP_by_elev = mean_cumP_by_elev.rename(columns={'Max_Cumulative_Precipitation': 'mean_max_cumulative_P'})

display(mean_cumP_by_elev)

,elev_class,mean_max_cumulative_P
0,0_500m,204.903442
1,1000_1500m,224.212136
2,1500_2000m,283.503016
3,2000_2500m,314.025267
4,500_1000m,189.457159


In [10]:
# Merge the mean back to the main DataFrame
onset_to_peak_mean_filtered = seasonal_maxcumP_meanSWE.merge(
    mean_cumP_by_elev,
    on='elev_class',
    how='left'
)

display(onset_to_peak_mean_filtered)

,Grid_ID,Seasonal_Year,lon,lat,elev_class,Max_Cumulative_Precipitation,Mean_SWE,mean_max_cumulative_P
0,1823,1980,-112.4088,49.0028,1000_1500m,233.992671,3.701920,224.212136
1,1823,1981,-112.4088,49.0028,1000_1500m,221.355045,12.164492,224.212136
2,1823,1982,-112.4088,49.0028,1000_1500m,130.209265,4.008709,224.212136
3,1823,1983,-112.4088,49.0028,1000_1500m,146.054043,5.082673,224.212136
4,1823,1984,-112.4088,49.0028,1000_1500m,218.806514,11.118445,224.212136
...,...,...,...,...,...,...,...,...
290615,13315,2019,-111.2250,59.9978,0_500m,209.251765,63.681722,204.903442
290616,13315,2020,-111.2250,59.9978,0_500m,222.117487,97.821268,204.903442
290617,13315,2021,-111.2250,59.9978,0_500m,198.002030,96.545600,204.903442
290618,13315,2022,-111.2250,59.9978,0_500m,142.264584,39.447457,204.903442


In [11]:
# Calculations to plot graphs
onset_to_peak_mean_filtered['SWE_P_ratio'] = (
    onset_to_peak_mean_filtered['Mean_SWE'] /
    onset_to_peak_mean_filtered['Max_Cumulative_Precipitation']
)

onset_to_peak_mean_filtered['Cum_P_anomaly'] = (
    onset_to_peak_mean_filtered['Max_Cumulative_Precipitation'] -
    onset_to_peak_mean_filtered['mean_max_cumulative_P']
)

display(onset_to_peak_mean_filtered)

,Grid_ID,Seasonal_Year,lon,lat,elev_class,Max_Cumulative_Precipitation,Mean_SWE,mean_max_cumulative_P,SWE_P_ratio,Cum_P_anomaly
0,1823,1980,-112.4088,49.0028,1000_1500m,233.992671,3.701920,224.212136,0.015821,9.780535
1,1823,1981,-112.4088,49.0028,1000_1500m,221.355045,12.164492,224.212136,0.054955,-2.857091
2,1823,1982,-112.4088,49.0028,1000_1500m,130.209265,4.008709,224.212136,0.030787,-94.002871
3,1823,1983,-112.4088,49.0028,1000_1500m,146.054043,5.082673,224.212136,0.034800,-78.158093
4,1823,1984,-112.4088,49.0028,1000_1500m,218.806514,11.118445,224.212136,0.050814,-5.405622
...,...,...,...,...,...,...,...,...,...,...
290615,13315,2019,-111.2250,59.9978,0_500m,209.251765,63.681722,204.903442,0.304331,4.348323
290616,13315,2020,-111.2250,59.9978,0_500m,222.117487,97.821268,204.903442,0.440403,17.214045
290617,13315,2021,-111.2250,59.9978,0_500m,198.002030,96.545600,204.903442,0.487599,-6.901412
290618,13315,2022,-111.2250,59.9978,0_500m,142.264584,39.447457,204.903442,0.277282,-62.638858


In [12]:
# take mean over Elevation_Category and Seasonal_Year
final_classification_data = (
    onset_to_peak_mean_filtered
    .groupby(['elev_class', 'Seasonal_Year'], as_index=False)
    .agg(
        Avg_SWE_P_ratio=('SWE_P_ratio', 'mean'),
        Avg_Cum_P_anomaly=('Cum_P_anomaly', 'mean')
    )
)
display(final_classification_data)

,elev_class,Seasonal_Year,Avg_SWE_P_ratio,Avg_Cum_P_anomaly
0,0_500m,1980,0.230944,-36.438050
1,0_500m,1981,0.217509,27.383216
2,0_500m,1982,0.273903,9.426227
3,0_500m,1983,0.163827,-34.078538
4,0_500m,1984,0.256816,24.902765
...,...,...,...,...
215,500_1000m,2019,0.126609,15.130074
216,500_1000m,2020,0.093493,-3.345060
217,500_1000m,2021,0.120038,28.954484
218,500_1000m,2022,0.190796,-47.410064


## Clean two-stage snow drought classification workflow

This version separates the workflow into two clear steps:

1. **Drought detection:** SWEI drought years are loaded first and merged with the SWE/P and precipitation-anomaly data.
2. **Drought type classification:** K-means is applied only to SWEI drought years, then cluster names are assigned automatically from the cluster centroids.

This avoids hard-coded cluster numbers and prevents drought years from being lost after clustering.

In [ ]:
# Load SWEI drought years
# Expected format: one row per elevation class, with a Drought_Years column containing either
# a Python-style list string, e.g. "[1982, 1995]", or a comma-separated string.
# normalize_years() lives in soyini.classification.
drought_years_df = pd.read_csv(drought_subset_dir)

# Make column names consistent across datasets/notebooks
if "Elevation_Category" in drought_years_df.columns and "elev_class" not in drought_years_df.columns:
    drought_years_df = drought_years_df.rename(columns={"Elevation_Category": "elev_class"})

required_cols = {"elev_class", "Drought_Years"}
missing_cols = required_cols - set(drought_years_df.columns)
if missing_cols:
    raise KeyError(f"drought_years_df is missing required columns: {missing_cols}")

drought_years_df["Drought_Years"] = drought_years_df["Drought_Years"].apply(normalize_years)

display(drought_years_df)

In [14]:
# Expand drought years to long format: one row per elevation class and drought year
# This is safer than keeping years as lists during the merge.
drought_years_long = (
    drought_years_df
    .explode("Drought_Years")
    .rename(columns={"Drought_Years": "Seasonal_Year"})
    .dropna(subset=["Seasonal_Year"])
    .copy()
)

drought_years_long["Seasonal_Year"] = drought_years_long["Seasonal_Year"].astype(int)
drought_years_long = drought_years_long[["elev_class", "Seasonal_Year"]].drop_duplicates()

display(drought_years_long.head())
print("Number of SWEI drought year/elevation combinations:", len(drought_years_long))

,elev_class,Seasonal_Year
0,0_500m,1983
0,0_500m,1992
0,0_500m,1997
0,0_500m,1998
0,0_500m,2000


Number of SWEI drought year/elevation combinations: 53


In [15]:
# Keep all classification variables, but mark which years are SWEI drought years.
classification_all_years = final_classification_data.copy()
classification_all_years["Seasonal_Year"] = classification_all_years["Seasonal_Year"].astype(int)

classification_all_years = classification_all_years.merge(
    drought_years_long.assign(is_swei_drought=True),
    on=["elev_class", "Seasonal_Year"],
    how="left"
)
classification_all_years["is_swei_drought"] = classification_all_years["is_swei_drought"].fillna(False)

# Only SWEI drought years go into the k-means classification.
drought_classification_data = classification_all_years[
    classification_all_years["is_swei_drought"]
].copy()

if drought_classification_data.empty:
    raise ValueError("No matching SWEI drought years found. Check elev_class names and Seasonal_Year values.")

# Check whether any drought years from the SWEI file were not found in the classification table.
matched_keys = set(zip(drought_classification_data["elev_class"], drought_classification_data["Seasonal_Year"]))
requested_keys = set(zip(drought_years_long["elev_class"], drought_years_long["Seasonal_Year"]))
missing_keys = sorted(requested_keys - matched_keys)

print("Matched SWEI drought year/elevation combinations:", len(matched_keys))
print("Missing combinations from classification data:", len(missing_keys))
if missing_keys:
    print("First missing combinations:", missing_keys[:20])

display(drought_classification_data.head())

Matched SWEI drought year/elevation combinations: 53
Missing combinations from classification data: 0


,elev_class,Seasonal_Year,Avg_SWE_P_ratio,Avg_Cum_P_anomaly,is_swei_drought
3,0_500m,1983,0.163827,-34.078538,True
12,0_500m,1992,0.114783,-38.881409,True
17,0_500m,1997,0.229115,-50.386696,True
18,0_500m,1998,0.192738,-41.978791,True
20,0_500m,2000,0.145435,-4.738631,True


In [ ]:
# Standardize variables within each elevation category, using drought years only.
# This makes the workflow transferable across Alberta, Bow River, or other datasets.
# zscore_by_group() lives in soyini.classification.
drought_classification_data = zscore_by_group(
    drought_classification_data,
    "elev_class",
    "Avg_Cum_P_anomaly",
    "cum_P_anomaly_z"
)

drought_classification_data = zscore_by_group(
    drought_classification_data,
    "elev_class",
    "Avg_SWE_P_ratio",
    "Avg_SWE_P_ratio_z"
)

# Remove any rows with missing/infinite features before clustering.
feature_cols = ["Avg_SWE_P_ratio_z", "cum_P_anomaly_z"]
drought_classification_data = drought_classification_data.replace([np.inf, -np.inf], np.nan)
drought_classification_data = drought_classification_data.dropna(subset=feature_cols).copy()

if len(drought_classification_data) < 3:
    raise ValueError("At least 3 drought-year rows are needed for 3-cluster k-means.")

display(drought_classification_data[["elev_class", "Seasonal_Year"] + feature_cols].head())

In [17]:
# K-means classification of drought years only.
# n_init is fixed for reproducibility; cluster labels are assigned from centroids below.
n_clusters = min(3, len(drought_classification_data))

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=50
)

drought_classification_data["cluster"] = kmeans.fit_predict(
    drought_classification_data[feature_cols]
)

centers = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=feature_cols
)
centers["cluster"] = np.arange(n_clusters)

print("Cluster centers in standardized drought-year space:")
display(centers)

Cluster centers in standardized drought-year space:


c:\Users\walimunige.rupasingh\OneDrive - University of Calgary\Documents\GitHub\Snow_Drought_Framework\.venv-1\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\walimunige.rupasingh\OneDrive - University of Calgary\Documents\GitHub\Snow_Drought_Framework\.venv-1\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


,Avg_SWE_P_ratio_z,cum_P_anomaly_z,cluster
0,-0.598989,-0.236215,0
1,-0.235442,1.130593,1
2,1.367636,-1.060549,2


In [ ]:
# Automatically assign physical snow-drought type names from cluster centroids.
# assign_cluster_names_from_centers() lives in soyini.classification. Logic:
# - Warm: relatively low SWE/P, relatively wet/normal precipitation anomaly
# - Dry: relatively high SWE/P, relatively dry precipitation anomaly
# - Warm & Dry: relatively low SWE/P and relatively dry precipitation anomaly
# This avoids hard-coding cluster numbers, which change between datasets.
cluster_name_map = assign_cluster_names_from_centers(centers)
drought_classification_data = drought_classification_data.merge(
    cluster_name_map,
    on="cluster",
    how="left"
)

print("Automatic cluster-name map:")
display(cluster_name_map)

display(
    drought_classification_data[
        ["elev_class", "Seasonal_Year", "Avg_SWE_P_ratio", "Avg_Cum_P_anomaly", "cluster", "cluster_name"]
    ].sort_values(["elev_class", "Seasonal_Year"])
)

In [19]:
# Add drought-type labels back to all years.
# SWEI drought years receive Warm, Dry, or Warm & Dry.
# Non-SWEI-drought years are Normal.
classified_drought_years = drought_classification_data[
    ["elev_class", "Seasonal_Year", "cluster_name", "cluster"]
].copy()

classification_all_years = classification_all_years.drop(columns=["cluster_name", "cluster"], errors="ignore")
classification_all_years = classification_all_years.merge(
    classified_drought_years,
    on=["elev_class", "Seasonal_Year"],
    how="left"
)
classification_all_years["cluster_name"] = classification_all_years["cluster_name"].fillna("Normal")
classification_all_years.loc[classification_all_years["is_swei_drought"] == False, "cluster"] = -1

# This check should be zero: every SWEI drought year should now have a drought type.
unclassified_droughts = classification_all_years[
    classification_all_years["is_swei_drought"] & classification_all_years["cluster_name"].isna()
]
print("Unclassified SWEI drought years:", len(unclassified_droughts))

display(classification_all_years.head())

Unclassified SWEI drought years: 0


,elev_class,Seasonal_Year,Avg_SWE_P_ratio,Avg_Cum_P_anomaly,is_swei_drought,cluster_name,cluster
0,0_500m,1980,0.230944,-36.438050,False,Normal,-1.0
1,0_500m,1981,0.217509,27.383216,False,Normal,-1.0
2,0_500m,1982,0.273903,9.426227,False,Normal,-1.0
3,0_500m,1983,0.163827,-34.078538,True,Warm & Dry,0.0
4,0_500m,1984,0.256816,24.902765,False,Normal,-1.0


In [ ]:
# Plot drought-year k-means classification in physical variable space.
# Colours + layout live in soyini.plotting.plot_classification_scatter.
from soyini.constants import ELEV_ORDER_ASC as elev_order

elevations_ordered = [e for e in elev_order if e in classification_all_years["elev_class"].unique()]

plot_classification_scatter(classification_all_years)
# To save: plot_classification_scatter(classification_all_years,
#     out_file=output_plots / "Snow_Drought_Classification_by_Elevation_clean.png")

In [ ]:
# Summary table: drought years separated by type, plus any unmatched/missing years.
# unmatched should be empty unless a SWEI drought year is missing from classification_all_years.

type_to_col = {
    "Dry": "dry",
    "Warm & Dry": "dry&warm",
    "Warm": "warm",
}

summary_rows = []
for elev in elevations_ordered:
    row = {"elev_class": elev}

    requested = set(
        drought_years_long.loc[drought_years_long["elev_class"] == elev, "Seasonal_Year"]
        .dropna()
        .astype(int)
        .tolist()
    )

    matched_for_elev = set(
        classification_all_years.loc[
            (classification_all_years["elev_class"] == elev) &
            (classification_all_years["is_swei_drought"]),
            "Seasonal_Year"
        ].dropna().astype(int).tolist()
    )

    assigned = set()
    for cname, col in type_to_col.items():
        years = classification_all_years.loc[
            (classification_all_years["elev_class"] == elev) &
            (classification_all_years["cluster_name"] == cname),
            "Seasonal_Year"
        ].dropna().astype(int).unique().tolist()
        years = sorted(years)
        row[col] = years
        assigned.update(years)

    row["Drought_Years"] = sorted(requested)
    row["unmatched"] = sorted(requested - assigned)
    row["missing_from_classification_data"] = sorted(requested - matched_for_elev)
    summary_rows.append(row)

sd_years_df = pd.DataFrame(summary_rows)
display(sd_years_df)

# Optional export
# sd_years_df.to_csv(output_dir / 'Snow_Drought_Years_by_Type_clean.csv', index=False)

In [ ]:
# Heatmap of final snow drought classification (Normal = non-SWEI-drought years).
# Colours + encoding live in soyini.plotting.plot_classification_heatmap.
plot_classification_heatmap(classification_all_years, elev_order=elev_order)
# To save: plot_classification_heatmap(classification_all_years, elev_order=elev_order,
#     out_file=output_plots / "Snow_Drought_Classification_Heatmap_clean.png")

## Method note for thesis/paper

This workflow treats snow drought typing as a two-stage process. First, SWEI is used to identify snow drought years for each elevation class. Then, only those SWEI drought years are classified into physical drought types using standardized SWE/P ratio and cumulative precipitation anomaly. Non-drought years are assigned as Normal after the clustering step. This prevents normal years from influencing the drought-type cluster boundaries and avoids hard-coded cluster IDs that change across datasets.